## chap 7. SNS 정보 수집

## step 1~3

In [164]:
#Step 1. 필요한 모듈과 라이브러리를 로딩합니다.
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import math
import os
import random
import unicodedata # 해시태그 수집 중 자음/모음 분리현상 방지용 모듈
import urllib.request
import urllib
import pandas as pd

#Step 2. 사용자에게 필요한 정보들을 입력 받기 #ID, 비밀번호
print("=" *80)
print("인스타그램 해쉬태그와 이미지 수집하기")
print("=" *80)

v_id = input("1.인스타그램의 ID를 입력하세요: ")
v_passwd = input("2.인스타그램의 비밀번호를 입력하세요: ")
query_txt = input("3.검색할 해쉬태그를 입력하세요(예: 강남맛집): ")

try :
    cnt = int( input('4.수집할 건수는 총 몇 건입니까?(기본값:10): '))
except ValueError :
    cnt = 10
    print('기본값인 10 건으로 수집을 진행합니다.')

page_cnt = math.ceil( cnt / 10)		#여기서 insta는 골치아픔
f_dir=input('5.파일이 저장될 경로만 쓰세요(기본경로 : c:\\py_temp\\ ) : ')
if f_dir =='' :
    f_dir = "c:\\py_temp\\"

#Step 3.결과를 저장할 폴더명과 파일명을 설정하고 폴더를 생성하기
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

sec_name='인스타그램'
img_dir = f_dir+s+'-'+query_txt+'-'+sec_name+'\\'+'image'

os.makedirs(img_dir)
os.chdir(img_dir)

fc_name=f_dir+s+'-'+query_txt+'-'+sec_name+'\\'+s+'-'+query_txt+'-'+sec_name+'.csv'
fx_name=f_dir+s+'-'+query_txt+'-'+sec_name+'\\'+s+'-'+query_txt+'-'+sec_name+'.xlsx'

인스타그램 해쉬태그와 이미지 수집하기


## step 4

In [165]:
# Step 4. 인스타그램 접속 후 자동 로그인 하기
s_time = time.time( )
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)
url = "https://www.instagram.com/"
driver.get(url)
driver.maximize_window()
time.sleep(random.randrange(1,5))

print("\n")
print("요청하신 데이터를 추출중이오니 잠시만 기다려 주세요~~~~^^")
print("\n")

#ID와 비번 입력후 로그인하기
eid = driver.find_element(By.NAME,'email')		#예전에는 username이었다가 email로 바뀜
for a in v_id :		#여기 한방에 입력하면 안들어가지기에 반복문 돌리는거임
    eid.send_keys(a)
    time.sleep(0.3)
epwd = driver.find_element(By.NAME,'pass')		#원래는 password였다가 pass로 바뀜
for b in v_passwd :
    epwd.send_keys(b)
    time.sleep(0.5)

# 로그인 정보 나중에 저장하기 클릭
driver.find_element(By.XPATH,'//*[@id="login_form"]/div/div[1]/div/div[3]/div/div/div').click()		#로그인 누름
time.sleep(7)		#로그인하고나서 좀 걸리기에 10초
driver.find_element(By.XPATH, "//div[@role='button']").click()	
time.sleep(6)
driver.find_element(By.XPATH, "//button[text()='나중에 하기']").click()
time.sleep(3)



요청하신 데이터를 추출중이오니 잠시만 기다려 주세요~~~~^^




## step 5~6

In [169]:
# Step 5. 검색할 키워드 입력하기
element = driver.find_element(By.XPATH,'//*[@id="mount_0_0_2s"]/div/div/div[2]/div/div/div[1]/div[1]/div[1]/div/div/div/div/div/div[2]/div/div[4]/span/div/a').click()
time.sleep(4)
element = driver.find_element(By.XPATH,'//*[@id="mount_0_0_2s"]/div/div/div[2]/div/div/div[1]/div[1]/div[1]/div/div/div/div/div/div/div/div[2]/div[2]/div/div/div[1]/div/div/input') #검색
for c in query_txt :
    element.send_keys(c)
    time.sleep(0.2)
time.sleep(3)
element.send_keys("\n")
element.send_keys("\n")
time.sleep(5)
# 인스타는 스크롤 내리면 다 사라지기에 **처음에 몇개 뽑을지 계산 잘해야함** (한번에 다 못 뽑음 - 한번에 30개정도 가능)
# 300개 뽑아야 하면 30개씩 계속 뽑아야 함
# 반드시 게시글 위에 커서 가져다놓고 마우스휠 돌려 글 뽑아내야함 *******

# Step 6. 전체 게시물의 원본 URL 추출하기
item=[ ] # 인스타그램 URL 주소 저장할 리스트
item2=[ ] # 중복값을 제거한 최종 URL 주소를 저장할 리스트

# 자동 스크롤다운 함수
def scroll_down(driver):
    driver.execute_script("window.scrollTo(0,document.body.scrollHeight);")
    time.sleep(5)

print('요청하신 데이터를 수집중이니 잠시만 기다려 주세요~^^')
print()

a = 1
while (a <= page_cnt):
    scroll_down(driver)

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    all_a = soup.find('div','x9f619').find_all('a')

    for i in all_a:
        url = i['href']
        item.append(url)
    item2 = pd.Series(item).drop_duplicates()

    if len(item2) >= cnt :
        break
    a += 1
    print('요청하신 데이터를 수집중이니 잠시만 더 기다려 주세요~^^')

# 추출된 URL 사용하여 전체 URL 완성하기
full_url=[ ]
url_cnt = 1
print('= 수집될 인스타그램 주소는 아래와 같습니다 =========')
for x in item2 :
    url = 'https://www.instagram.com' + x #앞에 인스타 주소를 꼭 붙여야함(생략된 것이기에)
    full_url.append(url)
    print(url_cnt,':',url)

    if url_cnt > cnt:
        break
    url_cnt += 1
print('========================================')
print()

요청하신 데이터를 수집중이니 잠시만 기다려 주세요~^^

= 수집될 인스타그램 주소는 아래와 같습니다 =========
1 : https://www.instagram.com/tasty.gangnam/
2 : https://www.instagram.com/kangnam11/
3 : https://www.instagram.com/all.about.gangnam.seongsu/
4 : https://www.instagram.com/songpa99/
5 : https://www.instagram.com/_mukpunzel/



## step 7

In [170]:
#Step 7. 각 페이지별로 이미지와 해쉬태그를 수집하기
count = 1 # 추출 데이터 건수 세기
no2= [ ] # 번호 저장
url2=[ ] # 수집완료된 url 저장
hash2 = [ ] # 해쉬 태그 저장

count = 1

for c in full_url :
    print()
    driver.get(c)
    time.sleep(random.randrange(3,9))

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    tags = soup.find('div','xt0psk2')

    try :
        tags_1 = tags.find_all('a')
    except :
        continue
    else :
        print('%s번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~' %count)
        print('게시물 URL:' , c )
        no2.append(count)
        url2.append(c)

        #해당 페이지의 대표 이미지 수집
        img_src = soup.find('div','_aagv').find('img')['src']
        urllib.request.urlretrieve(img_src , str(count)+'.jpg')
        print(img_dir,'아래에 %s번째 이미지 저장 완료===' %count)

        # 해당 페이지의 해시태그 수집
        # 비트맵 이미지 아이콘을 위한 대체 딕셔너리를 만들기
        import sys
        bmp_map = dict.fromkeys(range(0x10000, sys.maxunicode + 1), 0xfffd)		#자음 모음 막는 것

        hash_tags=[]
        for d in tags_1 :
            tags = d.get_text()
            tags_11 = tags.translate(bmp_map)		#자음 모음 막는 것
            tags_2 = unicodedata.normalize('NFC', tags_11)		#자음 모음 막는 것

            if tags_2[0:1]=='#':		#해쉬태그는 무조건 '#'으로 시작하기에.
                hash_tags.append(tags_2)

        print(hash_tags)
        hash2.append(hash_tags) # 각 게시물의 해시태그를 리스트 형태로 저장하기

        count += 1


1번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~
게시물 URL: https://www.instagram.com/tasty.gangnam/
c:\py_temp\2026-03-09-13-58-14-강남맛집-인스타그램\image 아래에 1번째 이미지 저장 완료===
[]

2번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~
게시물 URL: https://www.instagram.com/kangnam11/
c:\py_temp\2026-03-09-13-58-14-강남맛집-인스타그램\image 아래에 2번째 이미지 저장 완료===
[]

3번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~
게시물 URL: https://www.instagram.com/all.about.gangnam.seongsu/
c:\py_temp\2026-03-09-13-58-14-강남맛집-인스타그램\image 아래에 3번째 이미지 저장 완료===
[]

4번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~
게시물 URL: https://www.instagram.com/songpa99/
c:\py_temp\2026-03-09-13-58-14-강남맛집-인스타그램\image 아래에 4번째 이미지 저장 완료===
[]

5번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~
게시물 URL: https://www.instagram.com/_mukpunzel/
c:\py_temp\2026-03-09-13-58-14-강남맛집-인스타그램\image 아래에 5번째 이미지 저장 완료===
[]


## step 8

In [172]:
#Step 8. 수집된 해시태그를 csv , xls 형식으로 저장하기
# xls , csv로 저장하기 위해 데이터 프레임 생성하기
insta = pd.DataFrame( )
insta['번호'] = no2
insta['URL주소'] = url2
insta['해쉬태그'] = pd.Series(hash2)

# csv 형태로 저장하기
insta.to_csv(fc_name,encoding="utf-8-sig",index=False)

# 엑셀 형태로 저장하기
insta.to_excel(fx_name ,index=False , engine='openpyxl')

#Step 9. 요약 정보 출력하기
e_time = time.time( )
t_time = e_time - s_time

print("=" *120)
print("1.총 소요시간: %s 초" %round(t_time,1))
print("2.총 저장 건수: %s 건 " %count)
print("3.csv파일 저장 경로: %s" %fc_name)
print("4.xls파일 저장 경로: %s" %fx_name)
print("5.이미지파일 저장 경로: %s" %img_dir)
print("=" *120)
driver.close( )

1.총 소요시간: 1125.3 초
2.총 저장 건수: 6 건 
3.csv파일 저장 경로: c:\py_temp\2026-03-09-13-58-14-강남맛집-인스타그램\2026-03-09-13-58-14-강남맛집-인스타그램.csv
4.xls파일 저장 경로: c:\py_temp\2026-03-09-13-58-14-강남맛집-인스타그램\2026-03-09-13-58-14-강남맛집-인스타그램.xlsx
5.이미지파일 저장 경로: c:\py_temp\2026-03-09-13-58-14-강남맛집-인스타그램\image


InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff626cdaa55
	0x7ff626a38630
	0x7ff6267cd546
	0x7ff626818113
	0x7ff6268518e2
	0x7ff62684bb11
	0x7ff62684adc3
	0x7ff626796505
	0x7ff626d07810
	0x7ff626d01afd
	0x7ff626d22c1a
	0x7ff626a53345
	0x7ff626a5b81c
	0x7ff62679536a
	0x7ff626e62b08
	0x7ff8a0bce8d7
	0x7ff8a248c40c


## 원본 코드

In [ ]:
#Step 1. 필요한 모듈과 라이브러리를 로딩합니다.
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import math
import os
import random
import unicodedata # 해시태그 수집 중 자음/모음 분리현상 방지용 모듈
import urllib.request
import urllib
import pandas as pd

#Step 2. 사용자에게 필요한 정보들을 입력 받기 #ID, 비밀번호
print("=" *80)
print("인스타그램 해쉬태그와 이미지 수집하기")
print("=" *80)

v_id = input("1.인스타그램의 ID를 입력하세요: ")
v_passwd = input("2.인스타그램의 비밀번호를 입력하세요: ")
query_txt = input("3.검색할 해쉬태그를 입력하세요(예: 강남맛집): ")

try :
    cnt = int( input('4.수집할 건수는 총 몇 건입니까?(기본값:10): '))
except ValueError :
    cnt = 10
    print('기본값인 10 건으로 수집을 진행합니다.')

page_cnt = math.ceil( cnt / 10) #여기서 insta는 골치아픔
f_dir=input('5.파일이 저장될 경로만 쓰세요(기본경로 : c:\\py_temp\\ ) : ')
if f_dir =='' :
    f_dir = "c:\\py_temp\\"

#Step 3.결과를 저장할 폴더명과 파일명을 설정하고 폴더를 생성하기
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

sec_name='인스타그램'
img_dir = f_dir+s+'-'+query_txt+'-'+sec_name+'\\'+'image'

os.makedirs(img_dir)
os.chdir(img_dir)

fc_name=f_dir+s+'-'+query_txt+'-'+sec_name+'\\'+s+'-'+query_txt+'-'+sec_name+'.csv'
fx_name=f_dir+s+'-'+query_txt+'-'+sec_name+'\\'+s+'-'+query_txt+'-'+sec_name+'.xlsx'

# Step 4. 인스타그램 접속 후 자동 로그인 하기
s_time = time.time( )
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)
url = "https://www.instagram.com/"
driver.get(url)
time.sleep(random.randrange(1,5))

print("\n")
print("요청하신 데이터를 추출중이오니 잠시만 기다려 주세요~~~~^^")
print("\n")

#ID와 비번 입력후 로그인하기
eid = driver.find_element(By.NAME,'email') #예전에는 username이었다가 email로 바뀜
for a in v_id : #여기 한방에 입력하면 안들어가지기에 반복문 돌리는거임
    eid.send_keys(a)
    time.sleep(0.3)
epwd = driver.find_element(By.NAME,'pass') #원래는 password였다가 pass로 바뀜
for b in v_passwd :
    epwd.send_keys(b)
    time.sleep(0.5)

# 로그인 정보 나중에 저장하기 클릭
driver.find_element(By.XPATH,'//*[@id="login_form"]/div/div[1]/div/div[3]/div/div/div').click() #로그인 누름
time.sleep(10) #로그인하고나서 좀 걸리기에 10초
# '로그인 정보 저장 알림' 뜨기에 나중에 하기 버튼 클릭 XPATH해야됨
# '알림정보'도 뜨기에 나중에 하기 버튼 클릭 XPATH해야됨

# 알림정보 나중에 하기도 필요하면 추가하세요.


# Step 5. 검색할 키워드 입력하기
element = driver.find_element(By.XPATH,'//*[@id="react-root"]/section/nav/div[2]/div/div/div[2]/input') #검색
for c in query_txt :
    element.send_keys(c)
    time.sleep(0.2)
time.sleep(3)
element.send_keys("\n")
element.send_keys("\n")
time.sleep(5)
# 인스타는 스크롤 내리면 다 사라지기에 **처음에 몇개 뽑을지 계산 잘해야함** (한번에 다 못 뽑음 - 한번에 30개정도 가능)
# 300개 뽑아야 하면 30개씩 계속 뽑아야 함
# 반드시 게시글 위에 커서 가져다놓고 마우스휠 돌려 글 뽑아내야함 *******

# Step 6. 전체 게시물의 원본 URL 추출하기
item=[ ] # 인스타그램 URL 주소 저장할 리스트
item2=[ ] # 중복값을 제거한 최종 URL 주소를 저장할 리스트

# 자동 스크롤다운 함수
def scroll_down(driver):
    driver.execute_script("window.scrollTo(0,document.body.scrollHeight);")
    time.sleep(5)

print('요청하신 데이터를 수집중이니 잠시만 기다려 주세요~^^')
print()

a = 1
while (a <= page_cnt):
    scroll_down(driver)

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    all_a = soup.find('article','KC1QD').find_all('a')

    for i in all_a:
        url = i['href']
        item.append(url)
    item2 = pd.Series(item).drop_duplicates()

    if len(item2) >= cnt :
        break
    a += 1
    print('요청하신 데이터를 수집중이니 잠시만 더 기다려 주세요~^^')

# 추출된 URL 사용하여 전체 URL 완성하기
full_url=[ ]
url_cnt = 1
print('= 수집될 인스타그램 주소는 아래와 같습니다 =========')
for x in item2 :
    url = 'https://www.instagram.com' + x #앞에 인스타 주소를 꼭 붙여야함(생략된 것이기에)
    full_url.append(url)
    print(url_cnt,':',url)

    if url_cnt > cnt:
        break
    url_cnt += 1
print('========================================')
print()

#Step 7. 각 페이지별로 이미지와 해쉬태그를 수집하기
count = 1 # 추출 데이터 건수 세기
no2= [ ] # 번호 저장
url2=[ ] # 수집완료된 url 저장
hash2 = [ ] # 해쉬 태그 저장

count = 1

for c in full_url :
    print()
    driver.get(c)
    time.sleep(random.randrange(3,9))

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    tags = soup.find('div','EtaWk')

    try :
        tags_1 = tags.find_all('a')
    except :
        continue
    else :
        print('%s번째 게시물의 대표 이미지와 해쉬태그를 수집합니다~~~' %count)
        print('게시물 URL:' , c )
        no2.append(count)
        url2.append(c)

        #해당 페이지의 대표 이미지 수집
        img_src = soup.find('div','KL4Bh').find('img')['src']
        urllib.request.urlretrieve(img_src , str(count)+'.jpg')
        print(img_dir,'아래에 %s번째 이미지 저장 완료===' %count)

        # 해당 페이지의 해시태그 수집
        # 비트맵 이미지 아이콘을 위한 대체 딕셔너리를 만들기
        import sys
        bmp_map = dict.fromkeys(range(0x10000, sys.maxunicode + 1), 0xfffd) #자음 모음 막는 것

        hash_tags=[]
        for d in tags_1 :
            tags = d.get_text()
            tags_11 = tags.translate(bmp_map) #자음 모음 막는 것
            tags_2 = unicodedata.normalize('NFC', tags_11) #자음 모음 막는 것

            if tags_2[0:1]=='#': #해쉬태그는 무조건 '#'으로 시작하기에.
                hash_tags.append(tags_2)

        print(hash_tags)
        hash2.append(hash_tags) # 각 게시물의 해시태그를 리스트 형태로 저장하기

        count += 1

#Step 8. 수집된 해시태그를 csv , xls 형식으로 저장하기
# xls , csv로 저장하기 위해 데이터 프레임 생성하기
insta = pd.DataFrame( )
insta['번호'] = no2
insta['URL주소'] = url2
insta['해쉬태그'] = pd.Series(hash2)

# csv 형태로 저장하기
insta.to_csv(fc_name,encoding="utf-8-sig",index=False)

# 엑셀 형태로 저장하기
insta.to_excel(fx_name ,index=False , engine='openpyxl')

#Step 9. 요약 정보 출력하기
e_time = time.time( )
t_time = e_time - s_time

print("=" *120)
print("1.총 소요시간: %s 초" %round(t_time,1))
print("2.총 저장 건수: %s 건 " %count)
print("3.csv파일 저장 경로: %s" %fc_name)
print("4.xls파일 저장 경로: %s" %fx_name)
print("5.이미지파일 저장 경로: %s" %img_dir)
print("=" *120)
driver.close( )